In [15]:
# =====================================================================
# 0. SETUP: data, helpers, base case (Assignment 1)
# =====================================================================
# necessary imports
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from scipy.stats import norm
from load_industries import load_industry_returns, estimation_inputs      # ADAPT to your loader

# want to run the load_industries.py file to get the needed data
R = load_industry_returns("30_Industry_Portfolios.csv")
mu, Sigma, names = estimation_inputs(R, start="2015-01", end="2024-12")

#print(mu, Sigma, names)
      # MONTHLY returns (120, 30), in DECIMALS (not %)
T, n = R.shape
Sigma = (Sigma + Sigma.T) / 2                    # guard against tiny asymmetry
rf = 0.0                                         # risk-free rate (chosen = 0)
z = norm.ppf(0.95)                               # 1.645, used only in the 5% VaR check

status = {}
def solve(problem, var, name):
    problem.solve()
    status[name] = problem.status
    if var.value is None:                        # infeasible / failed
        return None
    return np.clip(var.value, 0, None)

# --- base case: max return s.t. variance <= equal-weight variance -------
w = cp.Variable(n)
w_eq = np.ones(n) / n
base_cons = [cp.sum(w) == 1, w >= 0,
             cp.quad_form(w, Sigma) <= w_eq @ Sigma @ w_eq]
w_base = solve(cp.Problem(cp.Maximize(mu @ w), base_cons), w, "base")

# --- "good return" floor: at least 80% of the base-case return ----------
r_floor = 0.8 * (mu @ w_base)
USE_FLOOR = True


# =====================================================================
# WISH 1 - CHAIR (a): original Sharpe formulation (NOT convex)
# =====================================================================
# max (mu'w - rf) / sqrt(w'Sigma w) is quasiconcave but not concave,
# so CVXPY rejects it. We only build it to show is_dcp() is False.
sharpe_direct = cp.Problem(
    cp.Maximize((mu @ w - rf) / cp.sqrt(cp.quad_form(w, Sigma))),
    [cp.sum(w) == 1, w >= 0])
dcp_direct = sharpe_direct.is_dcp()              # expected: False (cite in report)


# =====================================================================
# WISH 1 - CHAIR (b): convex reformulation (y = w/(mu'w - rf), kappa = 1/(mu'w - rf))
# =====================================================================
# Sharpe is scale-invariant, so fix the excess return to 1 and minimise variance:
#   min y'Sigma y   s.t.  mu'y - rf*kappa = 1,  1'y = kappa,  y, kappa >= 0
# then recover the portfolio as w = y / kappa.
y = cp.Variable(n)
kappa = cp.Variable()
sharpe_cons = [mu @ y - rf * kappa == 1,         # risk normalisation
               cp.sum(y) == kappa,               # budget constraint in scaled form
               y >= 0, kappa >= 0]
if USE_FLOOR:
    # mu'w >= r_floor  <=>  mu'y >= r_floor * kappa   (linear in y, kappa)
    sharpe_cons.append(mu @ y >= r_floor * kappa)
y_val = solve(cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)), sharpe_cons), y, "sharpe")
w_sharpe = None if y_val is None else y_val / kappa.value


# =====================================================================
# WISH 4 - RISK MANAGER (a): average of the worst 5% of months loses <= 8%
# =====================================================================
# 5% of 120 months = 6 months. CVaR = average of the k largest losses (-R w).
# sum_largest of an affine vector is convex, so this is a convex constraint.
k = int(round(0.05 * T))
losses = -R.values @ wcvar

common = [cp.sum(w) == 1, w >= 0]
if USE_FLOOR:
    common.append(mu @ w >= r_floor)
cvar_cons = [cp.sum_largest(losses, k) / k <= 0.08]

w_cvar = solve(cp.Problem(cp.Maximize(mu @ w), common + cvar_cons), w, "CVaR")

# =====================================================================
# WISH 4 - RISK MANAGER (b): average of the worst 5% of months loses <= 8%
# =====================================================================
# 5% of 120 months = 6 months. CVaR = average of the k largest losses (-R w).
# sum_largest of an affine vector is convex, so this is a convex constraint.
k = int(round(0.05 * T))

wcvar = cp.Variable(n, nonneg=True)
common = [cp.sum(wcvar) == 1, wcvar >= 0]
cvar_cons = [cp.sum_largest(losses, 6) / 6 <= 0.08]

w_cvar = solve(cp.Problem(cp.Maximize(mu @ wcvar), common + cvar_cons), w, "CVaR w/o lower")



# =====================================================================
# OUTPUTS (same definitions for every portfolio, as in Appendix A)
# =====================================================================
def metrics(wv):
    rp = R @ wv                                           # monthly portfolio returns
    wealth = np.concatenate([[1], np.cumprod(1 + rp)])    # W_0 = 1, W_1..W_T
    ret, vol = mu @ wv, np.sqrt(wv @ Sigma @ wv)
    return {
        "return %":   100 * ret,
        "vol %":      100 * vol,
        "ratio":      ret / vol,
        "max_dd %":   100 * np.max(1 - wealth / np.maximum.accumulate(wealth)),
        "worst12m %": 100 * min(np.prod(1 + rp[i:i + 12]) - 1 for i in range(T - 11)),
        "bad6 %":     100 * (-np.sort(rp)[:k]).mean(),    # avg of 6 worst months, as loss
        "5%VaR %":    100 * (ret - z * vol),              # normal-approx. 5% annual cutoff
        "max_w":      wv.max(),
        "n_eff":      1 / np.sum(wv ** 2),
    }

ports = {"base": w_base, "sharpe": w_sharpe, "CVaR": w_cvar}

print("\n--- Solver status (check every one!) ---")
for name, st in status.items():
    print(f"{name:14s} {st}")
print(f"\nOriginal Sharpe problem accepted by CVXPY (is_dcp)? {dcp_direct}")
print(f"Direct nonconvex solver (SLSQP) converged? {res.success}")

print("\n--- Metrics per portfolio ---")
table = {nm: metrics(wv) for nm, wv in ports.items() if wv is not None}
cols = list(next(iter(table.values())).keys())
print(f"{'':14s}" + "".join(f"{c:>12s}" for c in cols))
for nm, m in table.items():
    print(f"{nm:14s}" + "".join(f"{m[c]:12.2f}" for c in cols))

print("\n--- Risk manager's wish satisfied? (bad6 <= 8%) ---")
for nm, m in table.items():
    print(f"{nm:14s} {int(m['bad6 %'] <= 8 + 1e-6)}")

print("\n--- Top 5 weights per portfolio ---")
for nm, wv in ports.items():
    if wv is not None:
        top = np.argsort(-wv)[:5]
        print(f"{nm:14s}", ", ".join(f"#{i}:{wv[i]:.2f}" for i in top))


--- Solver status (check every one!) ---
base           optimal
sharpe         optimal
CVaR           optimal
CVaR w/o lower optimal

Original Sharpe problem accepted by CVXPY (is_dcp)? False
Direct nonconvex solver (SLSQP) converged? True

--- Metrics per portfolio ---
                  return %       vol %       ratio    max_dd %  worst12m %      bad6 %     5%VaR %       max_w       n_eff
base                 19.23       17.36        1.11       79.91      -62.76       12.49       -9.33        0.60        2.39
sharpe               19.45       17.56        1.11       80.02      -62.94       12.59       -9.44        0.62        2.27
CVaR                 26.02       41.58        0.63       90.28      -75.96       17.68      -42.38        1.00        1.00

--- Risk manager's wish satisfied? (bad6 <= 8%) ---
base           0
sharpe         0
CVaR           0

--- Top 5 weights per portfolio ---
base           #22:0.60, #21:0.17, #2:0.14, #19:0.06, #27:0.02
sharpe         #22:0.62, #21:0.1

In [ ]:
# ==============================================================================
# 1. METRICS COMPUTATION FUNCTION (Appendix A Specification)
# ==============================================================================
def compute_assignment2_metrics(w, mu, Sigma, R, portfolio_name="Portfolio"):
    """
    Computes all 8 required key numbers from Appendix A of Assignment 2.
    
    Parameters:
    -----------
    w     : np.ndarray (n,)
            Portfolio weights vector.
    mu    : np.ndarray (n,)
            Expected annual return vector (from estimation_inputs).
    Sigma : np.ndarray (n, n)
            Annual covariance matrix (from estimation_inputs).
    R     : np.ndarray or pd.DataFrame (T, n)
            Monthly returns matrix for Jan 2015 - Dec 2024 (T = 120).
    """
    w = np.asarray(w).flatten()
    
    # 1. Expected annual return mu^T w
    rec_return = float(mu @ w) * 100.0
    
    # 2. Annual volatility sqrt(w^T Sigma w)
    rec_volatility = float(np.sqrt(w @ Sigma @ w)) * 100.0
    
    # 3. Sharpe ratio without risk-free rate
    rec_ratio = rec_return / rec_volatility if rec_volatility > 0 else np.nan
    
    # Historical monthly portfolio returns series r_t^T w
    R_mat = np.asarray(R)
    monthly_returns = R_mat @ w
    T = len(monthly_returns)
    
    # 4. Maximum Drawdown: max_t (1 - W_t / max_{s <= t} W_s) with W_0 = 1
    W = np.cumprod(1.0 + monthly_returns)
    running_max = np.maximum.accumulate(W)
    drawdowns = 1.0 - (W / running_max)
    rec_max_drawdown = float(np.max(drawdowns)) * 100.0
    
    # 5. Worst compounded return over 12 consecutive months
    compounded_12m = np.array([
        np.prod(1.0 + monthly_returns[t : t + 12]) - 1.0 
        for t in range(T - 11)
    ])
    rec_worst_12m = float(np.min(compounded_12m)) * 100.0
    
    # 6. Average of the 6 worst monthly returns as a positive loss
    sorted_returns = np.sort(monthly_returns)
    worst_6_returns = sorted_returns[:6]
    rec_bad_months = float(-np.mean(worst_6_returns)) * 100.0
    
    # 7. Largest asset weight max_i w_i
    rec_max_weight = float(np.max(w)) * 100.0
    
    # 8. Effective number of industries 1 / sum_i w_i^2
    rec_n_eff = float(1.0 / np.sum(w ** 2))
    
    print("=" * 55)
    print(f" Key Metrics: {portfolio_name}")
    print("=" * 55)
    print(f"rec_return       : {rec_return:6.2f}%")
    print(f"rec_volatility   : {rec_volatility:6.2f}%")
    print(f"rec_ratio        : {rec_ratio:6.2f}")
    print(f"rec_max_drawdown : {rec_max_drawdown:6.2f}%")
    print(f"rec_worst_12m    : {rec_worst_12m:6.2f}%")
    print(f"rec_bad_months   : {rec_bad_months:6.2f}%")
    print(f"rec_max_weight   : {rec_max_weight:6.2f}%")
    print(f"rec_n_eff        : {rec_n_eff:6.2f}")
    print("=" * 55 + "\n")
    
    return {
        "rec_return": round(rec_return, 2),
        "rec_volatility": round(rec_volatility, 2),
        "rec_ratio": round(rec_ratio, 2),
        "rec_max_drawdown": round(rec_max_drawdown, 2),
        "rec_worst_12m": round(rec_worst_12m, 2),
        "rec_bad_months": round(rec_bad_months, 2),
        "rec_max_weight": round(rec_max_weight, 2),
        "rec_n_eff": round(rec_n_eff, 2)
    }

# ==============================================================================
# 2. SETUP BENCHMARKS AND CONSTANTS
# ==============================================================================
# Uses variables already loaded in your environment: mu, Sigma, R, names, wa
T, n = R.shape
r_bar = float(mu @ wa)          # Base portfolio expected annual return
min_good_return = 0.8 * r_bar   # "Good return" threshold (within 20% of base)
r_f = 0.0                       # Selected risk-free rate

print(f"Base Return (r_bar): {r_bar * 100:.2f}%")
print(f"Good Return Constraint Bound: {min_good_return * 100:.2f}%\n")


# ==============================================================================
# 3. METHOD 1: CHAIR'S WISH (Max Sharpe Ratio)
# ==============================================================================

# --- (A) Original Formulation (Non-Convex / Linear-Fractional) ---
print("--- [Chair] Attempting Original Formulation ---")
w_chair_orig = cp.Variable(n)
obj_chair_orig = cp.Maximize((mu @ w_chair_orig - r_f) / cp.sqrt(cp.quad_form(w_chair_orig, Sigma)))
constraints_chair_orig = [
    cp.sum(w_chair_orig) == 1,
    w_chair_orig >= 0,
    mu @ w_chair_orig >= min_good_return
]
prob_chair_orig = cp.Problem(obj_chair_orig, constraints_chair_orig)

try:
    prob_chair_orig.solve()
    print("Status:", prob_chair_orig.status)
except cp.error.DCPError as e:
    print("Status: REJECTED by CVXPY")
    print(f"DCP Explanation: {e}\n")


# --- (B) Reformulated Formulation (Convex via Charnes-Cooper) ---
print("--- [Chair] Solving Reformulated Problem (Charnes-Cooper) ---")
y = cp.Variable(n)
kappa = cp.Variable(nonneg=True)

# Minimize denominator variance subject to normalized numerator
prob_chair_reform = cp.Problem(
    cp.Minimize(cp.quad_form(y, Sigma)),
    [
        mu @ y - r_f * kappa == 1,
        cp.sum(y) == kappa,
        y >= 0,
        mu @ y >= min_good_return * kappa
    ]
)
prob_chair_reform.solve()

print("Status:", prob_chair_reform.status)
if prob_chair_reform.status == cp.OPTIMAL:
    w_chair = np.clip(y.value / kappa.value, 0, None)
    w_chair /= np.sum(w_chair)
    metrics_chair = compute_assignment2_metrics(
        w_chair, mu, Sigma, R, portfolio_name="Chair's Max Sharpe Portfolio"
    )


# ==============================================================================
# 4. METHOD 2: RISK MANAGER'S WISH (CVaR <= 8%)
# ==============================================================================
print("--- [Risk Manager] Solving Rockafellar-Uryasev CVaR Problem ---")

# Convert R to a clean 2D numpy array
R_mat = R.values if hasattr(R, "values") else np.asarray(R)
T, n = R_mat.shape
k = int(0.05 * T)  # k = 6 months for T = 120

w_risk = cp.Variable(n)
alpha = cp.Variable()
z = cp.Variable(T, nonneg=True)

# Define problem
constraints_risk = [
    cp.sum(w_risk) == 1,
    w_risk >= 0,
    mu @ w_risk >= min_good_return,  # Check for "good" return
    z >= -R_mat @ w_risk - alpha,
    alpha + (1.0 / k) * cp.sum(z) <= 0.08,
]

prob_risk = cp.Problem(cp.Maximize(mu @ w_risk), constraints_risk)
prob_risk.solve()

print("Status with 'good return' constraint:", prob_risk.status)

metrics_risk = None

# If infeasible with the 'good return' constraint, solve without it
# to see what the Risk Manager's wish actually achieves:
if prob_risk.status != cp.OPTIMAL:
  print("--> Infeasible! The fund cannot achieve a 'good' return (>= 80% of base)")
  print("    while keeping worst 5% monthly average loss <= 8%.")
  print("--> Re-solving without the return constraint to find feasible CVaR portfolio...")

  constraints_risk_relaxed = [
      cp.sum(w_risk) == 1,
      w_risk >= 0,
      z >= -R_mat @ w_risk - alpha,
      alpha + (1.0 / k) * cp.sum(z) <= 0.08,
  ]
  prob_risk_relaxed = cp.Problem(
      cp.Maximize(mu @ w_risk), constraints_risk_relaxed
  )
  prob_risk_relaxed.solve()
  print("Status without 'good return' constraint:", prob_risk_relaxed.status)

  if prob_risk_relaxed.status == cp.OPTIMAL:
    w_risk_opt = np.clip(w_risk.value, 0, None)
    w_risk_opt /= np.sum(w_risk_opt)
    metrics_risk = compute_assignment2_metrics(
        w_risk_opt,
        mu,
        Sigma,
        R_mat,
        portfolio_name="Risk Manager's CVaR Portfolio (Relaxed Return)",
    )
else:
  w_risk_opt = np.clip(w_risk.value, 0, None)
  w_risk_opt /= np.sum(w_risk_opt)
  metrics_risk = compute_assignment2_metrics(
      w_risk_opt,
      mu,
      Sigma,
      R_mat,
      portfolio_name="Risk Manager's CVaR Portfolio",
  )

# ==============================================================================
# 5. EXPORT READY VALUES FOR keynumbers.csv
# ==============================================================================
# Pick the portfolio you recommend to the board:
# (If you recommend the Chair's Sharpe portfolio, use metrics_chair;
#  if you recommend the Risk Manager's, use metrics_risk)
recommended_metrics = (
    metrics_chair if metrics_risk is None else metrics_risk
)  # or metrics_chair

print("\n>>> CSV-READY ENTRIES FOR keynumbers.csv <<<")
if recommended_metrics is not None:
  for key, val in recommended_metrics.items():
    print(f"{key},{val}")
else:
  print("No optimal portfolio found to export!")